# codingStandard — Google Colab Validation

Clean-runtime validation for the selected GitHub repository.

Enter a full GitHub URL, `owner/repository`, or owner name. Public repositories clone without credentials. If the repository is private, the notebook asks for a GitHub Personal Access Token securely only after the unauthenticated clone fails.


In [ ]:
from pathlib import Path
import getpass
import importlib.util
import json
import os
import platform
import shutil
import subprocess
import sys

os.chdir("/content")
DEFAULT_REPO_URL = "https://github.com/eaglesjo/codingStandard.git"
EXAMPLE_REPO_URL = "https://github.com/your-github-username/codingStandard.git"
DEFAULT_REPO_NAME = "codingStandard"
REPO = Path("/content/codingStandard")
RESULTS = Path("/content/codingstandard-colab-results.json")

def _colab_secret(name: str):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value.strip() if value else None
    except Exception:
        return None

def _normalize_repository(value: str) -> str:
    value = value.strip().rstrip("/")
    if not value:
        return DEFAULT_REPO_URL
    if value.startswith(("https://github.com/", "http://github.com/")):
        return value if value.endswith(".git") else value + ".git"
    if value.startswith("github.com/"):
        return "https://" + value + ("" if value.endswith(".git") else ".git")
    if value.count("/") == 1:
        return "https://github.com/" + value + ("" if value.endswith(".git") else ".git")
    if value.replace("-", "").replace("_", "").isalnum():
        return f"https://github.com/{value}/{DEFAULT_REPO_NAME}.git"
    raise ValueError("Enter a GitHub URL, owner/repository, or GitHub owner name.")

configured_url = os.environ.get("CODINGSTANDARD_REPO_URL", "").strip()
if configured_url:
    REPO_URL = _normalize_repository(configured_url)
else:
    prompt = input(
        f"GitHub repository, owner/repository, or owner\n[{EXAMPLE_REPO_URL}]: "
    ).strip()
    REPO_URL = DEFAULT_REPO_URL if not prompt else _normalize_repository(prompt)

def _clone(env=None):
    os.chdir("/content")
    if REPO.exists():
        shutil.rmtree(REPO)
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO)],
        capture_output=True, text=True, env=env, cwd="/content", check=False,
    )
    return proc

def _clone_with_token(token: str):
    env = os.environ.copy()
    askpass = Path("/tmp/codingstandard-git-askpass.sh")
    askpass.write_text(
        '#!/bin/sh\ncase "$1" in\n  *Username*) echo x-access-token ;;\n  *) printf "%s" "$GITHUB_TOKEN" ;;\nesac\n',
        encoding="utf-8",
    )
    askpass.chmod(0o700)
    env["GITHUB_TOKEN"] = token
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"
    try:
        return _clone(env)
    finally:
        askpass.unlink(missing_ok=True)

def clone_repository():
    # Public repository: no credential prompt.
    proc = _clone()
    if proc.returncode == 0:
        return "public"

    # Only ask for a token after unauthenticated access fails.
    token = os.environ.get("GITHUB_TOKEN") or _colab_secret("GITHUB_TOKEN")
    if not token:
        token = getpass.getpass(
            "Private GitHub repository detected or public clone failed.\n"
            "GitHub Personal Access Token (leave blank to stop): "
        ).strip()

    if not token:
        detail = (proc.stderr or proc.stdout or "unknown git clone error").strip()
        raise RuntimeError(f"Git clone failed without credentials: {detail}")

    auth_proc = _clone_with_token(token)
    if auth_proc.returncode != 0:
        detail = (auth_proc.stderr or auth_proc.stdout or "unknown git clone error").strip()
        raise RuntimeError(f"Git clone failed with token (exit {auth_proc.returncode}): {detail}")
    return "private/authenticated"

auth_mode = clone_repository()
os.chdir(REPO)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True, cwd=str(REPO)).strip()
print("Repository URL:", REPO_URL)
print("Authentication:", auth_mode)
print("Commit:", commit)
print("Local path:", REPO)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())


In [ ]:
if importlib.util.find_spec("psutil") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "psutil"], check=True)
if importlib.util.find_spec("torch") is None:
    raise RuntimeError("PyTorch is not available. Enable a Colab runtime with PyTorch installed.")

import psutil
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM free: {free / 1024**3:.2f} GB / {total / 1024**3:.2f} GB")
mem = psutil.virtual_memory()
print(f"RAM available: {mem.available / 1024**3:.2f} GB / {mem.total / 1024**3:.2f} GB")


In [ ]:
def run_step(label, command):
    print(f"\n=== {label} ===")
    proc = subprocess.run(command, capture_output=True, text=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode != 0:
        if proc.stderr:
            print(proc.stderr, file=sys.stderr)
        raise RuntimeError(f"{label} failed with exit code {proc.returncode}")
    return proc

profile_path = Path("/content/colab-environment-profile.json")
run_step("Environment profiler", [sys.executable, "LLM/environment.py", str(profile_path)])
profile = json.loads(profile_path.read_text(encoding="utf-8"))
print(json.dumps(profile, indent=2, ensure_ascii=False))


In [ ]:
llm_json = Path("/content/colab-llm-smoke.json")
run_step("LLM memory smoke test", [sys.executable, "LLM/memory_smoke_test.py", "--steps", "2", "--batch-size", "1", "--min-ram-free-gb", "0.25", "--min-vram-free-gb", "0.10", "--json", str(llm_json)])
llm_result = json.loads(llm_json.read_text(encoding="utf-8"))
print(json.dumps(llm_result, indent=2, ensure_ascii=False))

vision_json = Path("/content/colab-vision-smoke.json")
run_step("Vision memory smoke test", [sys.executable, "VISION/memory_smoke_test.py", "--device", "auto", "--image-size", "64", "--batch-size", "1", "--steps", "2", "--output", str(vision_json)])
vision_result = json.loads(vision_json.read_text(encoding="utf-8"))
print(json.dumps(vision_result, indent=2, ensure_ascii=False))


In [ ]:
run_step("Repository validation", [sys.executable, "scripts/validate.py"])

summary = {
    "platform": "google_colab",
    "repository_url": REPO_URL,
    "authentication": auth_mode,
    "commit": commit,
    "python": sys.version.split()[0],
    "pytorch": torch.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "environment_profile": str(profile_path),
    "llm_smoke_test": llm_result,
    "vision_smoke_test": vision_result,
    "status": "passed",
}
RESULTS.write_text(json.dumps(summary, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(summary, indent=2, ensure_ascii=False))
print("Saved:", RESULTS)
